In [1]:
# Install required packages first
# pip install git+https://github.com/huggingface/transformers accelerate
# pip install qwen-vl-utils[decord]==0.0.8

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

# Load model and processor
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_name)

# Prepare video input - you can use local file, URL, or list of frames
# Option 1: Ask for timestamps directly
# messages = [
#     {
#         "role": "user",
#         "content": [
#             {
#                 "type": "video",
#                 "video": "file:///path/to/surveillance_video.mp4",
#                 "fps": 1.0,  # Sample 1 frame per second
#             },
#             {"type": "text", "text": "Analyze this surveillance video and identify all instances of violent activity. For each violent action, specify the approximate time it occurs and describe what happens."},
#         ],
#     }
# ]

# Option 2: More structured prompt
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "/home/atin-ct3/action_recognition/data/Test-Data/YTDown.com_YouTube_Dramatic-Video-Domestic-Violence-Suspect_Media_RGMA9vllXmY_001_1080p.mp4",
                "fps": 1.0,
            },
            {"type": "text", "text": """Watch this video carefully and provide:
1. A list of all violent actions detected
2. The timestamp (in seconds or minutes) when each action occurs
3. A brief description of each violent incident

Format your response as:
- Timestamp: [time]
- Classification: [crowd fighting / dual combat / single person bully violence / suicide ]
- Description: [brief description of the incident] 
                         """},
        ],
    }
]

# # Option 3: Direct question format
# messages = [
#     {
#         "role": "user",
#         "content": [
#             {
#                 "type": "video",
#                 "video": "file:///path/to/surveillance_video.mp4",
#                 "fps": 1.0,
#             },
#             {"type": "text", "text": "At what times does violence occur in this video? List each incident with its timestamp and description."},
#         ],
#     }
# ]

# # Option 4: Multi-turn conversation for refinement
# messages = [
#     {
#         "role": "user",
#         "content": [
#             {
#                 "type": "video",
#                 "video": "file:///path/to/surveillance_video.mp4",
#                 "fps": 1.0,
#             },
#             {"type": "text", "text": "Is there any violent activity in this video?"},
#         ],
#     },
#     {
#         "role": "assistant",
#         "content": [
#             {"type": "text", "text": "Yes, I detected violent activity in the video."}
#         ]
#     },
#     {
#         "role": "user",
#         "content": [
#             {"type": "text", "text": "At what timestamps do these violent actions occur? Please list each incident."}
#         ]
#     }
# ]

# Prepare input
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# Important: Use return_video_kwargs=True for video processing
image_inputs, video_inputs, video_kwargs = process_vision_info(
    messages, return_video_kwargs=True
)

# Process inputs with video kwargs (includes fps information)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
    **video_kwargs,  # Important for video temporal alignment
)
inputs = inputs.to("cuda")

# Generate output
generated_ids = model.generate(**inputs, max_new_tokens=128)

# Trim input tokens from output
generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

# Decode output
output_text = processor.batch_decode(
    generated_ids_trimmed, 
    skip_special_tokens=True, 
    clean_up_tokenization_spaces=False
)

print(output_text[0])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
qwen-vl-utils using decord to read video.


- Timestamp: 0:00 - 0:05
- Classification: Single person bully violence
- Description: A man in a red jacket is seen walking towards another individual, who appears to be a child. The man in the red jacket then starts to push and shove the child, causing him to fall to the ground.

- Timestamp: 0:05 - 0:10
- Classification: Single person bully violence
- Description: The man in the red jacket continues to push and shove the child, who is now on the ground. He then stands over the child, appearing to taunt him
